
# STRATUM — Dataset Exploration

## Dataset
Advanced Process Control and Statistical Process Control Data for Thickness Prediction of AlCu and WTi Metal Layers in Semiconductor Manufacturing.

**Source:** Infineon Technologies dataset published on Zenodo (Version 2, October 2026).

**File:** apc_spc_AlCu_inputs_outputs.csv

## Objective
Explore the semiconductor AlCu Physical Vapor Deposition (PVD) dataset to understand process variables, wafer measurements, quality indicators, missing values, and potential machine-learning targets.

## Initial Tasks
1. Load the original dataset.
2. Verify its dimensions.
3. Preview the records.
4. Inspect the available features.
5. Identify data quality issues.


In [3]:
# Import libraries

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("STRATUM: Libraries imported successfully!")


STRATUM: Libraries imported successfully!


In [13]:
# STRATUM - Load the Infineon AlCu dataset

filename = "apc_spc_AlCu_inputs_outputs.csv"

# Locate the dataset from the project root or notebooks folder
current_dir = Path.cwd()

csv_path = current_dir / "data" / "raw" / filename

if not csv_path.exists():
    csv_path = current_dir.parent / "data" / "raw" / filename

if not csv_path.exists():
    raise FileNotFoundError(
        f"Dataset not found. Current directory: {current_dir}"
    )

# Load the original CSV
df = pd.read_csv(
    csv_path,
    sep=";",
    skiprows=1,
    low_memory=False
)

print("Dataset loaded successfully!")
print(f"Loaded dataset: {filename}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully!
Loaded dataset: apc_spc_AlCu_inputs_outputs.csv
Rows: 5208
Columns: 178


In [6]:
# Display selected columns from the first five records

columns_to_view = [
    "bat_id",
    "wafer_id",
    "lot",
    "equipment",
    "machine",
    "mean_thickness",
    "mean_resistance",
    "quality_overall"
]

df[columns_to_view].head()

,bat_id,wafer_id,lot,equipment,machine,mean_thickness,mean_resistance,quality_overall
0,13266763,V612511104,V7177247,equipment_13,machine_4,152.453686,0.002889,0
1,13267071,V612511105,V7177247,equipment_14,machine_4,152.247616,0.002887,0
2,13267105,V612511106,V7177247,equipment_1,machine_4,152.749744,0.002871,0
3,13274131,V612511116,V7177484,equipment_2,machine_3,151.925461,0.002880,0
4,13345497,V612515010,V7177985,equipment_13,machine_4,152.716449,0.002881,0


## 2. Dataset Structure and Feature Classification

The dataset is divided into wafer metadata, Advanced Process Control (APC) variables, Statistical Process Control (SPC) measurements, and process quality indicators.

We will identify these groups before preprocessing or model development.

In [7]:
#Categorise dataset columns

# 1. Wafer and equipment metadata
metadata_cols = [
    "bat_id", "wafer_id", "lot",
    "equipment", "machine",
    "batchstart", "batchend", "slot",
    "runstart", "runend"
]

# 2. Thickness measurements - 17 wafer points
thickness_cols = [
    f"RAW_VALUE_t_{i}" for i in range(1, 18)
]

# 3. Sheet resistance measurements - 17 points
resistance_cols = [
    f"RAW_VALUE_r_{i}" for i in range(1, 18)
]

# 4. Derived resistivity values - 17 points
resistivity_cols = [
    f"RAW_VALUE_{i}" for i in range(1, 18)
]

# 5. Wafer-level averages
average_cols = [
    "mean_thickness",
    "mean_resistance"
]

# 6. Stability and quality indicators
quality_cols = [
    "stable_state",
    "quality_thickness",
    "quality_resistance",
    "quality_resistivity",
    "quality_overall"
]

# Remaining columns are APC process variables
known_cols = set(
    metadata_cols +
    thickness_cols +
    resistance_cols +
    resistivity_cols +
    average_cols +
    quality_cols
)

apc_cols = [
    col for col in df.columns
    if col not in known_cols
]

# Verify every column is accounted for
assert len(known_cols) + len(apc_cols) == len(df.columns)

groups = {
    "Metadata": metadata_cols,
    "APC process variables": apc_cols,
    "Thickness measurements": thickness_cols,
    "Sheet resistance measurements": resistance_cols,
    "Derived resistivity": resistivity_cols,
    "Wafer-level averages": average_cols,
    "Stability and quality indicators": quality_cols
}

for group_name, columns in groups.items():
    print(f"{group_name}: {len(columns)} columns")

print("\nTotal:", sum(len(x) for x in groups.values()))


Metadata: 10 columns
APC process variables: 110 columns
Thickness measurements: 17 columns
Sheet resistance measurements: 17 columns
Derived resistivity: 17 columns
Wafer-level averages: 2 columns
Stability and quality indicators: 5 columns

Total: 178


## 3. Data Types

Inspect numerical, categorical and timestamp-related variables before preprocessing.

In [8]:
#Inspect data types

print("Dataset shape:", df.shape)

print("\nData type distribution:")
print(df.dtypes.value_counts())

print("\nSelected column data types:")

selected_cols = [
    "wafer_id",
    "lot",
    "equipment",
    "runstart",
    "POWER_TARGET_DEP1_A",
    "mean_thickness",
    "mean_resistance",
    "quality_overall"
]

print(df[selected_cols].dtypes)


Dataset shape: (5208, 178)

Data type distribution:
float64    163
str         11
int64        4
Name: count, dtype: int64

Selected column data types:
wafer_id                   str
lot                        str
equipment                  str
runstart                   str
POWER_TARGET_DEP1_A    float64
mean_thickness         float64
mean_resistance        float64
quality_overall          int64
dtype: object


## 4. Missing Value Analysis

Determine which dataset columns contain missing observations and calculate their missing-value percentages.

In [9]:
#Analyse missing values

missing_count = df.isnull().sum()

missing_percentage = (
    missing_count / len(df)
) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

# Sort from highest to lowest missing percentage
missing_summary = missing_summary.sort_values(
    "Missing Percentage",
    ascending=False
)

print("Columns with missing values:",
      (missing_count > 0).sum())

print("Columns without missing values:",
      (missing_count == 0).sum())

print("Columns with over 50% missing data:",
      (missing_percentage > 50).sum())

display(missing_summary.head(15).round(2))


Columns with missing values: 47
Columns without missing values: 131
Columns with over 50% missing data: 11


,Missing Count,Missing Percentage
WAFERCOUNT_RECIPE_MR,3571,68.57
VOLTS_TARGET_DEP1_G,3534,67.86
GAS2_DEP1_S,3382,64.94
WAFERCOUNT_MR,3205,61.54
LIFE_TARGET_RUN_I,3183,61.12
T_RUN_ENG,2837,54.47
PRESS_CHAMBER_DEP1_R,2822,54.19
PRESS_TRANSFER_RUN_X,2822,54.19
PRESS_IONGAUGE_BP_I_ENG,2729,52.40
PRESS_IONGAUGE_BP_I_ENG1,2633,50.56


## 5. Primary Prediction Target

The primary supervised learning target is mean_thickness.

We will investigate its distribution before selecting and evaluating regression models.

In [10]:
#Inspect the primary target

target = "mean_thickness"

print("Primary ML target:", target)

print("\nThickness statistics:")
display(df[target].describe().round(3))

print("\nMissing target values:",
      df[target].isnull().sum())


Primary ML target: mean_thickness

Thickness statistics:


count    5208.000
mean      152.094
std         2.485
min       141.962
25%       150.446
50%       152.177
75%       153.745
max       163.642
Name: mean_thickness, dtype: float64


Missing target values: 0


## 6. Wafer Quality Label Analysis

Examine the distribution of thickness, sheet resistance, resistivity, overall quality and process stability indicators.

In [11]:
#Analyse quality indicators

quality_columns = [
    "quality_thickness",
    "quality_resistance",
    "quality_resistivity",
    "quality_overall",
    "stable_state"
]

for col in quality_columns:
    print(f"\n--- {col} ---")

    counts = df[col].value_counts(dropna=False)
    percentages = (
        df[col].value_counts(
            normalize=True,
            dropna=False
        ) * 100
    ).round(2)

    result = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages
    })

    display(result)



--- quality_thickness ---


,Count,Percentage
quality_thickness,,
good,5196,99.77
bad,12,0.23



--- quality_resistance ---


,Count,Percentage
quality_resistance,,
good,5146,98.81
bad,62,1.19



--- quality_resistivity ---


,Count,Percentage
quality_resistivity,,
good,4913,94.34
bad,295,5.66



--- quality_overall ---


,Count,Percentage
quality_overall,,
0,4873,93.57
1,335,6.43



--- stable_state ---


,Count,Percentage
stable_state,,
1,4438,85.22
0,770,14.78


## 7. Manufacturing Metadata Analysis

Analyse wafer identifiers, production lots, equipment,
machines and processing timestamps.

These attributes will support equipment-level comparisons,
temporal analysis and leakage-aware train-test splitting.


In [12]:
# Explore manufacturing metadata

print("Unique wafer IDs:",
      df["wafer_id"].nunique())

print("Unique lots:",
      df["lot"].nunique())

print("Unique equipment IDs:",
      df["equipment"].nunique())

print("Unique machine IDs:",
      df["machine"].nunique())

# Convert timestamps temporarily for inspection
run_dates = pd.to_datetime(
    df["runstart"],
    errors="coerce"
)

print("\nEarliest process record:",
      run_dates.min())

print("Latest process record:",
      run_dates.max())

print("Invalid timestamps:",
      run_dates.isna().sum())

print("\nEquipment record distribution:")
display(df["equipment"].value_counts())


Unique wafer IDs: 5208
Unique lots: 2951
Unique equipment IDs: 16
Unique machine IDs: 6

Earliest process record: 2021-07-31 06:51:54
Latest process record: 2025-05-31 17:04:14
Invalid timestamps: 0

Equipment record distribution:


equipment
equipment_14    440
equipment_13    431
equipment_1     423
equipment_2     420
equipment_9     412
equipment_15    400
equipment_8     382
equipment_11    315
equipment_3     313
equipment_7     307
equipment_16    264
equipment_12    261
equipment_4     257
equipment_5     205
equipment_10    190
equipment_6     188
Name: count, dtype: int64

## 8. Machine Learning Feature Selection Notes

### Primary prediction task
Predict `mean_thickness` from suitable APC process measurements.

### Candidate model inputs
- Deposition power
- Chamber pressure
- Target current and voltage
- Gas-related parameters
- Temperature measurements
- Process durations
- Other relevant APC sensor variables

### Variables excluded from primary prediction inputs
- All 17 measured thickness values
- All 17 measured sheet resistance values
- All 17 derived resistivity values
- mean_thickness and mean_resistance
- Post-process quality indicators
- Wafer and batch identifiers used purely for traceability

### Additional monitoring tasks
- Thickness uniformity analysis
- Sheet resistance and resistivity analysis
- Wafer quality classifications
- Equipment-level process comparisons
- Process stability monitoring

### Important
Measured SPC outputs will be analysed separately from virtual metrology predictions.

Data cleaning, feature selection and model training have not yet been performed.